In [ ]:
import sys
import numpy as np
import scipy
import torch

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Critical compatibility check
from scipy.linalg import cholesky

print("\nNumPy and SciPy imports successful!")

In [ ]:
%pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    evaluate \
    huggingface_hub

In [1]:

# ============================================================
# PHASE 6: QLoRA FINE-TUNING
# Centralized Imports
# ============================================================

# -------------------------
# Standard Python Libraries
# -------------------------
import os
import gc
import json
import time
import random
import warnings

from pathlib import Path

# -------------------------
# Data Processing
# -------------------------
import numpy as np
import pandas as pd

# -------------------------
# Visualization
# -------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# PyTorch
# -------------------------
import torch
import torch.nn as nn

# -------------------------
# Hugging Face Datasets
# -------------------------
from datasets import Dataset

# -------------------------
# Transformers
# -------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed
)

# -------------------------
# PEFT / LoRA
# -------------------------
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

# -------------------------
# TRL / Supervised Fine-Tuning
# -------------------------
from trl import SFTTrainer, SFTConfig

# -------------------------
# Evaluation
# -------------------------
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# -------------------------
# Progress Monitoring
# -------------------------
from tqdm.auto import tqdm

print("All Phase 6 imports completed successfully!")


All Phase 6 imports completed successfully!


In [2]:

import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("========== ENVIRONMENT ==========")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)

print("\n========== GPU ==========")

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )


========== ENVIRONMENT ==========
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
Accelerate: 1.15.0
PEFT: 0.21.2
BitsAndBytes: 0.50.2
TRL: 1.18.1

========== GPU ==========
CUDA Available: True
GPU: Tesla T4
Total GPU Memory: 14.56 GB


In [4]:
# Load the training dataset
DIR_PATH = Path("/kaggle/input/datasets/dgowda11/sms-spam-cleaned")

TRAIN_PATH = DIR_PATH / "train.csv"
TEST_PATH = DIR_PATH / "test.csv"

assert TRAIN_PATH.exists(), "train path not found !"
assert TEST_PATH.exists(), "test path not found"

train_df  = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Tranining Shape :",train_df.shape)
print("Testing Shape :",test_df.shape)

display(train_df.head())

Tranining Shape : (4135, 2)
Testing Shape : (1034, 2)


,label,message
0,ham,Cool. So how come you havent been wined and di...
1,ham,Huh so late... Fr dinner?
2,ham,my ex-wife was not able to have kids. Do you w...
3,ham,Hope ur head doesn't hurt 2 much ! Am ploughin...
4,ham,Ryder unsold.now gibbs.
